In [ ]:
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
#回归问题评价指标
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error 
from sklearn.model_selection import cross_val_score
#分类问题评价指标
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss#对数损失（交叉熵损失）
random_seed = 123
np.random.seed(random_seed)
data=pd.read_excel('DATA.xlsx',header=None)
y=data.iloc[:,-1].values
# X=pd.read_csv('pca_X.csv',index_col=0).values
X=data.iloc[:,:-1].values
# 将数据分为训练集和测试集
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)
print('X_train.shape',X_train.shape)
print('X_test.shape',X_test.shape)
print('y_train.shape',y_train.shape)
print('y_test.shape',y_test.shape)
svr=SVR()

# param_distributions ={
#     'C': [0.001,0.01,0.1,1,10,100],
#     'kernel': ['linear', 'poly', 'rbf','sigmoid'],
# }
param_distributions = {
    'C': [0.1, 1, 10, 100],  # 正则化参数
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],  # 核函数参数
    'kernel': ['linear', 'rbf', 'sigmoid'],  # 核函数类型    #注poly核函数跑的话特别消耗时间
}
#param_distributions=[{'kernel': ['rbf'], 'gamma': [1e-3, 1e-4],'C': [1, 10, 100, 1000]},{'kernel': ['linear'], 'C': [1, 10, 100, 1000]},{'kernel':['poly'], 'gamma': [1e-3, 1e-4],'C': [1, 10, 100, 1000]},{'kernel':['sigmoid'], 'gamma': [1e-3, 1e-4],'C': [1, 10, 100, 1000]}]

random_search = RandomizedSearchCV(svr, param_distributions, n_iter=10, cv=5,verbose=2)
random_search.fit(X_train,y_train)

#获取最佳参数
best_params=random_search.best_params_
print("网格搜索的最佳参数：",best_params)

best_model = random_search.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_test=best_model.predict(X_test)

r2_score_train=best_model.score(X_train,y_train)
mse_train = mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_score_test=best_model.score(X_test,y_test)
mse_test = mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test) 

print("训练集R2",r2_score_train)
print("训练集均方根值：",mse_train)
print("训练集MAE：", mae_train) 
print("测试集R2",r2_score_test)
print("测试集均方根值：",mse_test)
print("测试集MAE：", mae_test)  # 输出测试集MAE